# mKdV Soliton Discovery via Physics-Informed Neural Network

This notebook trains a neural network to discover the soliton solution of the **modified Korteweg–de Vries (mKdV) equation**:

$$u_t + 6u^2 u_x + u_{xxx} = 0$$

The exact soliton solution with wavenumber $k$ is:

$$u(x,t) = k^2 \operatorname{sech}\!\bigl(k(x - k^4 t)\bigr)$$

We use $k = 1$, giving $u(x,t) = \operatorname{sech}(x - t)$ — a unit-amplitude soliton traveling at speed $c = k^4 = 1$.

The network learns $u(x,t)$ with **no labeled training data** — only physics constraints.
Unlike the Duffing notebook, mKdV has no free PDE coefficients to discover; the network purely learns the solution shape.

## 0. Imports & Reproducibility

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pylab as pl

# Fix random seeds for reproducibility
np.random.seed(37)
torch.manual_seed(42)

device = torch.device('cpu')
print(f"Using device: {device}")

## 1. Create the Input Data

The problem is now 2-dimensional: we sample random $(x, t)$ pairs over a space–time domain
$[-L_x,\; L_x + c\,T] \times [0, T]$ to use as collocation points for the PDE residual.

Boundary-condition points are sampled at $t = 0$ (the initial profile) and split into:
- **Inside** — $|x| < x_{\text{edge}}$ (near the peak, enforces shape constraints)
- **Outside** — $|x| \geq x_{\text{edge}}$ (tail region, enforces decay)

where $x_{\text{edge}} = \operatorname{arccosh}(3) \approx 1.763$ is the point where $\operatorname{sech}(x_{\text{edge}}) = 1/3$.

In [ ]:
n_data = 2000       # collocation points for PDE residual
n_bc   = 500        # boundary-condition points (t = 0 slice)
L_x    = 5.0        # spatial half-width
T      = 1.0        # time horizon
c_true = 1.0        # true wave speed  (k=1, c = k^4 = 1)

# Random (x, t) pairs covering the soliton's space-time trajectory
x_col = np.random.uniform(-L_x, L_x + c_true * T, size=n_data)
t_col = np.random.uniform(0, T, size=n_data)

# Boundary-condition x values at t = 0
x_bc   = np.random.uniform(-L_x, L_x, size=n_bc)
x_edge = np.arccosh(3.0)   # sech(x_edge) = 1/3  ->  arccosh(3) ~ 1.763

mask_in = np.abs(x_bc) < x_edge
x_in    = x_bc[ mask_in]
x_out   = x_bc[~mask_in]

print(f"Collocation points : {len(x_col)}")
print(f"BC inside  (|x| < {x_edge:.3f}): {len(x_in)}")
print(f"BC outside (|x| >= {x_edge:.3f}): {len(x_out)}")
print(f"\nTrue soliton : u(x,t) = sech(x - {c_true:.0f}t)   (unit amplitude, speed = {c_true})")

In [ ]:
f = pl.figure(figsize=(10, 5))
ax = f.add_subplot(1, 1, 1)

ax.scatter(x_col, t_col,              s=1, color=(1, 0, 0), alpha=0.3, label='Collocation (PDE)')
ax.scatter(x_in,  np.zeros_like(x_in), s=8, color=(0, 0.8, 0), label=f'BC inside  (|x| < {x_edge:.2f})')
ax.scatter(x_out, np.zeros_like(x_out), s=8, color=(0, 0, 1), label='BC outside')

ax.set_title('Training Locations in Space-Time', fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('t')
ax.legend()
pl.tight_layout()
pl.show()

Convert arrays to PyTorch tensors. A helper `at_t0` stacks 1D $x$ values with $t=0$ into $(N, 2)$ tensors for the network.

In [ ]:
# Collocation tensor — shape (n_data, 2)
xt_col_T = torch.tensor(
    np.stack([x_col, t_col], axis=1), dtype=torch.float32
)

# BC tensors (1D; use at_t0 to build (x, 0) pairs inside residual functions)
x_in_T  = torch.tensor(x_in,  dtype=torch.float32)
x_out_T = torch.tensor(x_out, dtype=torch.float32)


def at_t0(x_1d):
    """Stack a 1D x tensor with t=0 into an (N, 2) input tensor."""
    x = x_1d.unsqueeze(1)           # (N, 1)
    t = torch.zeros_like(x)         # (N, 1)
    return torch.cat([x, t], dim=1) # (N, 2)


print("xt_col_T shape :", xt_col_T.shape)
print("at_t0(x_in_T)  :", at_t0(x_in_T).shape)
print("at_t0(x_out_T) :", at_t0(x_out_T).shape)

## 2. Define the Neural Network

Two hidden layers with 64 neurons each and Tanh activation.  The network is wider and deeper
than the Duffing version because:
- The input is now 2D $(x, t)$
- The mKdV PDE involves a third-order spatial derivative, requiring more expressive capacity

- **Input**: $(x,\, t)$
- **Output**: scalar $u(x, t)$

In [ ]:
class Network(nn.Module):

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 64)    # 2 inputs: (x, t)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 1)
        self.activation = nn.Tanh()

    def forward(self, xt):
        h = self.activation(self.fc1(xt))
        h = self.activation(self.fc2(h))
        return self.fc3(h)


u_NN = Network().to(device)
u_NN

## 3. Physics Residuals (Loss Terms)

The loss is a weighted sum of five physics-based residuals.
Constraints L1–L4 are enforced at $t = 0$ (the initial profile); L5 is the PDE on all $(x, t)$ pairs.

| Term | Constraint |
|------|------------|
| L1   | $u(0,\,0) = u_{\text{peak}}$ — peak height at $t=0$ |
| L2   | $u_x(0,\,0) = 0$ — spatially flat peak at $t=0$ |
| L3   | $u(\pm x_{\text{edge}},\,0) = u_{\text{peak}}/3$ — initial width |
| L4   | $|u(x,\,0)| \leq u_{\text{peak}}/3$ for $|x| \geq x_{\text{edge}}$ — initial tail decay |
| L5   | $u_t + 6u^2 u_x + u_{xxx} = 0$ — mKdV PDE on all $(x,t)$ |

`gradients(y, x, dim)` computes $\partial y / \partial x_{[\text{dim}]}$ and keeps the graph
so it can be called three times in succession to reach $u_{xxx}$.

In [ ]:
def gradients(y, x, dim):
    """Compute dy/dx[:, dim] via autograd, retaining graph for higher-order calls."""
    g = torch.autograd.grad(
        y, x, grad_outputs=torch.ones_like(y),
        retain_graph=True, create_graph=True
    )[0]
    return g[:, dim:dim+1]


u_peak = 1.0


def center_peak_residual():
    """L1: u(0, 0) = u_peak"""
    xt0 = torch.tensor([[0.0, 0.0]])
    return u_NN(xt0) - u_peak


def flat_peak_residual():
    """L2: u_x(0, 0) = 0  (spatially flat peak at t=0)"""
    xt0 = torch.tensor([[0.0, 0.0]], requires_grad=True)
    u   = u_NN(xt0)
    return gradients(u, xt0, 0)   # partial w.r.t. x (dim=0)


def edge_peak_residual():
    """L3: u(+/-x_edge, 0) = u_peak / 3"""
    xe = torch.tensor([[-x_edge, 0.0], [+x_edge, 0.0]])
    return u_NN(xe) - u_peak / 3


def outside_bound_residual(x_out):
    """L4: penalise |u(x, 0)| > u_peak/3 in the tail region."""
    xt0 = at_t0(x_out)
    return F.relu(torch.abs(u_NN(xt0)) - u_peak / 3)


def ode_residual(xt):
    """L5: u_t + 6*u^2*u_x + u_xxx = 0  (mKdV PDE residual)."""
    xt_r  = xt.detach().requires_grad_(True)
    u     = u_NN(xt_r)
    u_x   = gradients(u,    xt_r, 0)   # du/dx
    u_t   = gradients(u,    xt_r, 1)   # du/dt
    u_xx  = gradients(u_x,  xt_r, 0)  # d2u/dx2
    u_xxx = gradients(u_xx, xt_r, 0)  # d3u/dx3
    return u_t + 6.0 * u**2 * u_x + u_xxx

## 4. Training Setup

- **Optimizer**: Adam, `lr = 1e-4` (network weights only — mKdV has no learnable PDE parameters)
- **Epochs**: 200 000
- **Loss weights**: `c1=c2=c3=c4=1`, `c5=50` (ODE term up-weighted)

> **Note**: Computing $u_{xxx}$ requires three sequential autograd calls, making each epoch
> more expensive than the Duffing (2nd-order) case.  Reduce `n_epochs` or `n_data` if runtime is a concern.

In [ ]:
n_epochs = 2 * 10**5
n_rep    = 5 * 10**3   # print every n_rep epochs

c1, c2, c3, c4, c5 = 1, 1, 1, 1, 50

# No learnable PDE parameters — optimise network weights only
optimizer = torch.optim.Adam(u_NN.parameters(), lr=1e-4)
mse       = nn.MSELoss()

# Pre-allocate history arrays
Epoch_List      = np.arange(n_epochs) + 1
Train_Loss_List = np.zeros(n_epochs)
Train_L1_List   = np.zeros(n_epochs)
Train_L2_List   = np.zeros(n_epochs)
Train_L3_List   = np.zeros(n_epochs)
Train_L4_List   = np.zeros(n_epochs)
Train_L5_List   = np.zeros(n_epochs)

print("Training configuration ready.")
print(f"  Epochs       : {n_epochs:,}")
print(f"  Loss weights : c1={c1}, c2={c2}, c3={c3}, c4={c4}, c5={c5}")
print(f"  LR           : 1e-4  (network weights only)")

## 5. Train the Neural Network

Each epoch computes all five residuals, accumulates the weighted loss, and back-propagates.
Progress is printed every 5 000 epochs.

In [ ]:
for ep in range(n_epochs):
    u_NN.train()
    optimizer.zero_grad()

    # L1 — peak height at t=0
    r1 = center_peak_residual()
    L1 = mse(r1, torch.zeros_like(r1))

    # L2 — flat spatial peak at t=0
    r2 = flat_peak_residual()
    L2 = mse(r2, torch.zeros_like(r2))

    # L3 — edge width at t=0
    r3 = edge_peak_residual()
    L3 = mse(r3, torch.zeros_like(r3))

    # L4 — tail bound at t=0
    r4 = outside_bound_residual(x_out_T)
    L4 = mse(r4, torch.zeros_like(r4))

    # L5 — mKdV PDE residual on all (x, t) collocation points
    r5 = ode_residual(xt_col_T)
    L5 = mse(r5, torch.zeros_like(r5))

    loss = c1*L1 + c2*L2 + c3*L3 + c4*L4 + c5*L5
    loss.backward()
    optimizer.step()

    Train_Loss_List[ep] = loss.item()
    Train_L1_List[ep]   = L1.item()
    Train_L2_List[ep]   = L2.item()
    Train_L3_List[ep]   = L3.item()
    Train_L4_List[ep]   = L4.item()
    Train_L5_List[ep]   = L5.item()

    if ep % n_rep == 0:
        print(f"Epoch {ep+1:>7,}  Loss: {Train_Loss_List[ep]:.6e}")

## 6. Loss Curves

Visualise how the total loss and each individual loss term evolved over training.

In [ ]:
f1, (sf1_1, sf1_2, sf1_3) = pl.subplots(1, 3, figsize=(24, 6))

# Total loss
sf1_1.plot(Epoch_List, Train_Loss_List, lw=3, label='Total Loss')
sf1_1.set_title('Total Training Loss', fontweight='bold')
sf1_1.set_xlabel('Epoch')
sf1_1.set_ylabel('Loss')
sf1_1.set_yscale('log')
sf1_1.legend()

# Individual (unweighted) loss terms
for data, label in [
    (Train_L1_List, 'L1 — peak height'),
    (Train_L2_List, 'L2 — flat peak'),
    (Train_L3_List, 'L3 — edge width'),
    (Train_L4_List, 'L4 — tail bound'),
    (Train_L5_List, 'L5 — mKdV residual'),
]:
    sf1_2.plot(Epoch_List, data, lw=3, label=label)
sf1_2.set_title('Individual Loss Terms (unweighted)', fontweight='bold')
sf1_2.set_xlabel('Epoch')
sf1_2.set_ylabel('Loss')
sf1_2.set_yscale('log')
sf1_2.legend()

# Weighted contributions
for data, c, label in [
    (Train_L1_List, c1, 'c1*L1'),
    (Train_L2_List, c2, 'c2*L2'),
    (Train_L3_List, c3, 'c3*L3'),
    (Train_L4_List, c4, 'c4*L4'),
    (Train_L5_List, c5, 'c5*L5'),
]:
    sf1_3.plot(Epoch_List, c * data, lw=3, label=label)
sf1_3.set_title('Weighted Loss Contributions', fontweight='bold')
sf1_3.set_xlabel('Epoch')
sf1_3.set_ylabel('Weighted Loss')
sf1_3.set_yscale('log')
sf1_3.legend()

pl.tight_layout()
pl.show()

## 7. Evaluate the Learned Solution

Compare the network output against the exact soliton $u(x,t) = \operatorname{sech}(x - t)$
at three time snapshots: $t = 0$, $0.5$, and $1.0$.

The soliton should translate to the right without changing shape — the network must learn
this travelling-wave behaviour purely from the initial-profile constraints and the PDE.

In [ ]:
n_val   = 1000
t_snaps = [0.0, 0.5, 1.0]
x_eval  = np.linspace(-L_x, L_x + c_true * T, n_val)

u_NN.eval()
fig, axes = pl.subplots(1, 3, figsize=(18, 5))

for ax, t_snap in zip(axes, t_snaps):
    xt_np = np.stack([x_eval, np.full_like(x_eval, t_snap)], axis=1)
    xt_T  = torch.tensor(xt_np, dtype=torch.float32)

    with torch.no_grad():
        u_pred = u_NN(xt_T).numpy().squeeze()

    u_exact = 1.0 / np.cosh(x_eval - c_true * t_snap)
    rmse    = np.sqrt(np.mean((u_pred - u_exact)**2))

    ax.plot(x_eval, u_pred,  color='g', lw=4, label='Neural Network')
    ax.plot(x_eval, u_exact, color='b', lw=4, ls=':', label=f'Exact: sech(x - {c_true:.0f}t)')
    ax.set_title(f't = {t_snap}   RMSE = {rmse:.5f}', fontweight='bold')
    ax.set_xlabel('x')
    ax.set_ylabel('u(x, t)')
    ax.legend()

pl.suptitle('mKdV Soliton: Neural Network vs Exact Solution', fontweight='bold', y=1.02)
pl.tight_layout()
pl.show()

In [ ]:
print("=== Solution Error Summary ===")
for t_snap in [0.0, 0.5, 1.0]:
    xt_np = np.stack([x_eval, np.full_like(x_eval, t_snap)], axis=1)
    xt_T  = torch.tensor(xt_np, dtype=torch.float32)
    with torch.no_grad():
        u_pred = u_NN(xt_T).numpy().squeeze()
    u_exact = 1.0 / np.cosh(x_eval - c_true * t_snap)
    rmse = np.sqrt(np.mean((u_pred - u_exact)**2))
    print(f"  t = {t_snap:.1f}  RMSE = {rmse:.6f}")

print(f"\n=== True soliton ===")
print(f"  u(x,t) = sech(x - t)")
print(f"  Wave speed : c = k^4 = 1   (k = 1)")
print(f"  Amplitude  : k^2 = 1")
print(f"  Width      : x_edge = arccosh(3) / k = {x_edge:.4f}")